In [1]:
# NumPy reshape()메서드 : 차원축소, 차원확대 메서드

import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, SimpleRNN, Dense

In [2]:
# 시계열 데이터를 예측 -> 학습에 사용하는 변수는 하나 -> 슬라이딩 윈도우 알고리즘을 사용해서 X n개, y 1개 분할 -> 인덱스 순서를 변경없이 사용해야 한다.

# 1. 주가 데이터 (가상 데이터)
price = np.array([
    100, 102, 101, 105, 107, 106, 110, 112, 115, 113,
    118, 120, 119, 123, 125, 128, 126, 130, 133, 135
], dtype=float)

print(price)
print(price.shape)

[100. 102. 101. 105. 107. 106. 110. 112. 115. 113. 118. 120. 119. 123.
 125. 128. 126. 130. 133. 135.]
(20,)


In [3]:
# 스케일링 -> 신경망 -> 무조건 한다 -> 왜? 큰 수보다 작은 수의 학습이 더 잘 이루어진다.
print(price.max())
price = price / price.max()
print(price) # 0.7 ~ 1 사이에 있으므로, min-max방식으로 정규화한다(0~1값을 취하기 위해)

price2 = (price - price.min()) / (price.max() - price.min())
print(price2) # 0 ~ 1 의 값이 위치하게 된다

135.0
[0.74074074 0.75555556 0.74814815 0.77777778 0.79259259 0.78518519
 0.81481481 0.82962963 0.85185185 0.83703704 0.87407407 0.88888889
 0.88148148 0.91111111 0.92592593 0.94814815 0.93333333 0.96296296
 0.98518519 1.        ]
[0.         0.05714286 0.02857143 0.14285714 0.2        0.17142857
 0.28571429 0.34285714 0.42857143 0.37142857 0.51428571 0.57142857
 0.54285714 0.65714286 0.71428571 0.8        0.74285714 0.85714286
 0.94285714 1.        ]


In [4]:
# sklearn api를 사용해도 된다.
# from sklearn.preprocessing import MinMaxScaler, MaxAbsScaler

In [5]:
# price 데이터셋(20,) -> X : (17, 3), y : (17,)

X, y = [], []   # 리스트
window_size = 3

In [6]:
for i in range(0, len(price2) - window_size):
    X.append(price[ i : i + window_size ])
    y.append(price[ i + window_size] )

print(X)
print(y)
# 리스트 -> NumPy배열로 변경
X = np.array(X)
y = np.array(y)

print(X.shape)
print(y.shape)

[array([0.74074074, 0.75555556, 0.74814815]), array([0.75555556, 0.74814815, 0.77777778]), array([0.74814815, 0.77777778, 0.79259259]), array([0.77777778, 0.79259259, 0.78518519]), array([0.79259259, 0.78518519, 0.81481481]), array([0.78518519, 0.81481481, 0.82962963]), array([0.81481481, 0.82962963, 0.85185185]), array([0.82962963, 0.85185185, 0.83703704]), array([0.85185185, 0.83703704, 0.87407407]), array([0.83703704, 0.87407407, 0.88888889]), array([0.87407407, 0.88888889, 0.88148148]), array([0.88888889, 0.88148148, 0.91111111]), array([0.88148148, 0.91111111, 0.92592593]), array([0.91111111, 0.92592593, 0.94814815]), array([0.92592593, 0.94814815, 0.93333333]), array([0.94814815, 0.93333333, 0.96296296]), array([0.93333333, 0.96296296, 0.98518519])]
[np.float64(0.7777777777777778), np.float64(0.7925925925925926), np.float64(0.7851851851851852), np.float64(0.8148148148148148), np.float64(0.8296296296296296), np.float64(0.8518518518518519), np.float64(0.837037037037037), np.float64

In [7]:
# RNN 모델에 입력 후 학습, 예측1
# RNN 입력층은 3차원을 입력받아서, 2차원씩 가져와 입력
# 2차원 X를 3치원으로 변경
X = X.reshape(17, 3, 1)
print(X.shape)

(17, 3, 1)


In [ ]:
# 모델 초기화
model = Sequential([
    Input(shape=(3,1)),
    SimpleRNN(10),    # RNN
    Dense(1)    # 지도 회귀 : 1, 지도 분류 : 클래스 수
])

In [ ]:
# 모델 컴파일
model.compile(optimizer='adam', loss='mse')

In [14]:
model.fit(X, y, epochs=100, verbose=0)

In [17]:
# 마지막 3일 다음날의 주가예측을 위한 X_test준비
X_test = price[-3:]
print(X_test)
print(X_test.shape) # 1차원 -> 3차원 (3,) -> (3,1) -> (1, 3, 1)

X_test = X_test.reshape(-1, 3, 1)   # (1, 3, 1)
print(X_test)
print(X_test.shape)

[0.96296296 0.98518519 1.        ]
(3,)
[[[0.96296296]
  [0.98518519]
  [1.        ]]]
(1, 3, 1)


In [ ]:
# 예측
pred = model.predict(X_test)
print(pred) # 값/최댓값135 한 정규화 값

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 115ms/step
[[0.88430095]]


In [19]:
print(pred[0][0] * 135)

119.38063
